# Broad all-dynamics, multi-topology, multi-seed single-shot experiment

This notebook is the **computation/cache layer** for the broad final benchmark.
It intentionally contains no publication plotting.

It combines both experiment regimes developed so far.

## A. Original unstructured benchmark

For **all six propagation models**:

- linear consensus (`laplacian`);
- COCA;
- Hegselmann--Krause;
- Friedkin--Johnsen;
- nonlinear influence;
- repulsion.

The graph is generated from the original Barabási--Albert family and the
opinions are a permutation of a population-wide spread from 0 to 1.

We use:

- topology seeds: `3, 4, 5, 6, 7`;
- opinion-permutation seeds: `0, 1, 2, 4, 5`.

Thus each dynamics model has 25 matched unstructured environments.

## B. Structured mechanism benchmark

We retain the already-fixed families that were discovered without using the
learned-controller result for topology selection:

### HK
- `ring_bridge`;
- `paired_bridge`;
- `dense_weak_bridge`.

### FJ and nonlinear influence
- `asymmetric_broadcast`;
- `dual_broadcast`;
- `split_broadcast`.

Each structured family has five mild environment perturbations.

## Repetitions

Every distinct environment is run with three independent learned-identifier
seeds. Baselines are evaluated once per distinct environment.

Total design:

- 150 unstructured environments = 6 dynamics x 25;
- 45 structured environments = 3 dynamics x 15;
- 195 distinct environments;
- 585 learned single-shot trajectories;
- 585 baseline trajectories (no control, uniform, true graph).

The three learned seeds are repetitions **within the same environment**.
Publication confidence intervals are therefore computed after first averaging
those repetitions within each distinct environment.

## Exploration

Campaign 0 remains passive. Campaigns 1--19 use pure learned exploitation.
Explicit random exploration is fixed to zero, based on the previous dedicated
exploration ablation.

In [1]:
from __future__ import annotations

import hashlib
import inspect
import json
import os
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 280)


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError(
        "Could not find repository root containing opinion_dynamics/."
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from rl_envs_forge.envs.network_graph.network_graph import NetworkGraph
from rl_envs_forge.envs.network_graph.graph_utils import (
    compute_eigenvector_centrality,
    compute_laplacian,
)
from opinion_dynamics.baseline import centrality_based_continuous_control
import opinion_dynamics.experiments.online_single_shot as online_single_shot_module

run_single_shot_online_identification = (
    online_single_shot_module.run_single_shot_online_identification
)

try:
    GIT_COMMIT = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except Exception:
    GIT_COMMIT = "unknown"

print("Repository root:", REPO_ROOT)
print("Git commit:", GIT_COMMIT)
print("online_single_shot:", inspect.getsourcefile(online_single_shot_module))
print("NetworkGraph:", inspect.getsourcefile(NetworkGraph))

C:\Users\Chainsword\AppData\Local\Temp\ipykernel_42216\3850186055.py:16: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Repository root: d:\Work\repos\RL\unknown_graph_networks
Git commit: ca3199a
online_single_shot: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\online_single_shot.py
NetworkGraph: c:\Users\Chainsword\anaconda3\envs\phd_rl_algos\Lib\site-packages\rl_envs_forge\envs\network_graph\network_graph.py


## Configuration

In [2]:
# Parallel worker mode.
NUM_SHARDS = 8
SHARD_ENV = "BROAD_FULL_SHARD_ID"
THREAD_ENV = "BROAD_FULL_TORCH_THREADS"

_shard_value = os.environ.get(SHARD_ENV)
WORKER_MODE = _shard_value is not None
SHARD_ID = int(_shard_value) if WORKER_MODE else None

if WORKER_MODE and SHARD_ID not in range(NUM_SHARDS):
    raise ValueError(
        f"{SHARD_ENV} must be in 0..{NUM_SHARDS - 1}; got {SHARD_ID}"
    )

TORCH_THREADS = int(os.environ.get(THREAD_ENV, "2"))
torch.set_num_threads(max(1, TORCH_THREADS))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

DEVICE = os.environ.get("BROAD_FULL_DEVICE", "cpu").lower()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable.")

print("Mode:", "worker" if WORKER_MODE else "analysis/check")
if WORKER_MODE:
    print("Shard:", f"{SHARD_ID + 1}/{NUM_SHARDS}")
print("Device:", DEVICE)
print("Torch threads:", torch.get_num_threads())

STUDY_NAME = "broad_all_dynamics_multitopology_multiseed"
PIPELINE_VERSION = "2026-08-17-v1"

N = 15
TARGET = 1.0

NUM_CAMPAIGNS = 20
T_CAMPAIGN = 0.5
T_S = 0.1

MAX_U = 0.20
TOTAL_CONTROLLED_BUDGET = 6.0
B_CAMPAIGN = TOTAL_CONTROLLED_BUDGET / (NUM_CAMPAIGNS - 1)

LAMBDA_MIX = 0.70
EXPLORATION_CAMPAIGNS = 0
EPSILON_SCHEDULE = [0.0] * NUM_CAMPAIGNS

FIT_LR = 1e-3
FIT_MAX_STEPS = 1_000
FIT_MAE_STOP = 5e-4
FIT_BATCH_SIZE = 256
FIT_CHECK_EVERY = 200
IDENTIFIER_KWARGS = {"hidden_dim": 16}

LEARNING_SEEDS = [0, 1, 2]

# Original generic/random benchmark seeds.
UNSTRUCTURED_TOPOLOGY_SEEDS = [3, 4, 5, 6, 7]
UNSTRUCTURED_OPINION_SEEDS = [0, 1, 2, 4, 5]

# Dynamics parameterization used in the generic benchmark.
UNSTRUCTURED_DYNAMICS = {
    "laplacian": {},
    "coca": {},
    "hegselmannkrause": {
        "hk_epsilon": 0.50,
        "hk_include_self": True,
    },
    "friedkinjohnsen": {
        "fj_lambda": 0.98,
    },
    "nonlinearinfluence": {
        "nonlinear_beta": 4.0,
    },
    "repulsion": {
        "repulsion_epsilon": 0.30,
        "repulsion_strength": 0.10,
    },
}

# Structured mechanism parameters.
STRUCTURED_HK_EPSILON = 0.25
STRUCTURED_FJ_LAMBDA = 0.98
STRUCTURED_NLI_BETA = 4.0

STRUCTURED_ENVIRONMENT_SEEDS = [0, 1, 2, 3, 4]

HK_FAMILIES = {
    "ring_bridge": {
        "peer_pattern": "ring",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.40,
        "bridge_weight": 3.0,
    },
    "paired_bridge": {
        "peer_pattern": "paired",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.55,
        "bridge_weight": 3.0,
    },
    "dense_weak_bridge": {
        "peer_pattern": "dense",
        "hub_leaf_weight": 3.0,
        "peer_weight": 0.08,
        "bridge_weight": 3.0,
    },
}

BROADCAST_FAMILIES = {
    "asymmetric_broadcast": {
        "kind": "asymmetric",
        "hub_influence_total": 0.85,
    },
    "dual_broadcast": {
        "kind": "dual",
        "hub_influence_total": 0.85,
    },
    "split_broadcast": {
        "kind": "split",
        "hub_influence_total": 0.85,
    },
}

FJ_HUB_LEVEL = 0.16
FJ_LEAF_LEVEL = 0.48
NLI_HUB_LEVEL = 0.10
NLI_LEAF_LEVEL = 0.50

EDGE_WEIGHT_JITTER_FRAC = 0.05
HK_OPINION_JITTER = 0.006
BROADCAST_OPINION_JITTER = 0.010

RESULTS_ROOT = (
    REPO_ROOT
    / "opinion_dynamics"
    / "experiments"
    / "results"
)
RESULTS_DIR = (
    RESULTS_ROOT
    / "experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CACHE_ROOT = RESULTS_ROOT / "_trial_cache" / STUDY_NAME
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("Campaign budget:", B_CAMPAIGN)
print("Uniform allocation/node:", B_CAMPAIGN / N)
print("Explicit exploration campaigns:", EXPLORATION_CAMPAIGNS)

Mode: analysis/check
Device: cpu
Torch threads: 2
Campaign budget: 0.3157894736842105
Uniform allocation/node: 0.021052631578947368
Explicit exploration campaigns: 0


## Preserve dynamics-specific settings when environments are cloned

In [3]:
def _copy_maybe(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return np.array(value, copy=True)
    if isinstance(value, list):
        return list(value)
    if isinstance(value, tuple):
        return tuple(value)
    return value


def patched_env_kwargs_from_env(
    env: Any,
    *,
    t_campaign: float | None = None,
    t_s: float | None = None,
) -> dict[str, Any]:
    n = int(env.num_agents)

    kwargs = dict(
        connectivity_matrix=np.array(env.connectivity_matrix, copy=True),
        num_agents=n,
        max_u=np.array(env.max_u, copy=True),
        desired_opinion=float(env.desired_opinion),
        t_campaign=float(env.t_campaign if t_campaign is None else t_campaign),
        t_s=float(env.t_s if t_s is None else t_s),
        dynamics_model=str(env.dynamics_model),
        initial_opinions=np.array(
            getattr(env, "initial_opinions", env.opinions),
            copy=True,
        ),
        control_resistance=np.array(
            getattr(env, "control_resistance", np.zeros(n)),
            copy=True,
        ),
        max_steps=int(getattr(env, "max_steps", 10_000)),
        opinion_end_tolerance=float(
            getattr(env, "opinion_end_tolerance", 0.01)
        ),
        control_beta=float(getattr(env, "control_beta", 0.4)),
        normalize_reward=bool(getattr(env, "normalize_reward", True)),
        terminal_reward=float(getattr(env, "terminal_reward", 0.0)),
        terminate_when_converged=bool(
            getattr(env, "terminate_when_converged", False)
        ),
        budget=float(getattr(env, "budget", 1000.0)),
        seed=(
            int(env.seed)
            if getattr(env, "seed", None) is not None
            else None
        ),
    )

    model = str(env.dynamics_model).lower()

    if model == "hegselmannkrause":
        kwargs["hk_epsilon"] = float(env.hk_epsilon)
        kwargs["hk_include_self"] = bool(env.hk_include_self)

    elif model == "friedkinjohnsen":
        kwargs["fj_lambda"] = _copy_maybe(env.fj_lambda)
        kwargs["fj_prejudice"] = _copy_maybe(env.fj_prejudice)

    elif model == "nonlinearinfluence":
        kwargs["nonlinear_beta"] = float(env.nonlinear_beta)

    elif model == "repulsion":
        kwargs["repulsion_epsilon"] = float(env.repulsion_epsilon)
        kwargs["repulsion_strength"] = float(env.repulsion_strength)

    return kwargs


def patched_make_env_from_template(
    env_template: Any,
    *,
    t_campaign: float | None = None,
    t_s: float | None = None,
) -> Any:
    return env_template.__class__(
        **patched_env_kwargs_from_env(
            env_template,
            t_campaign=t_campaign,
            t_s=t_s,
        )
    )


online_single_shot_module.env_kwargs_from_env = patched_env_kwargs_from_env
online_single_shot_module.make_env_from_template = patched_make_env_from_template

run_single_shot_online_identification = (
    online_single_shot_module.run_single_shot_online_identification
)

print("Environment cloning compatibility layer installed.")

Environment cloning compatibility layer installed.


## Common graph helpers

In [4]:
def _normalize_rows(raw: np.ndarray) -> np.ndarray:
    raw = np.asarray(raw, dtype=float)
    np.fill_diagonal(raw, 0.0)
    sums = raw.sum(axis=1, keepdims=True)
    if np.any(sums <= 0):
        raise RuntimeError("Every graph row must have positive mass.")
    A = raw / sums
    np.fill_diagonal(A, 0.0)
    return A


def _jitter_positive_edges(
    raw: np.ndarray,
    rng: np.random.Generator,
    frac: float,
) -> np.ndarray:
    out = np.asarray(raw, dtype=float).copy()
    mask = out > 0
    multipliers = rng.uniform(
        1.0 - float(frac),
        1.0 + float(frac),
        size=out.shape,
    )
    out[mask] *= multipliers[mask]
    np.fill_diagonal(out, 0.0)
    return out


def _add_undirected(
    raw: np.ndarray,
    i: int,
    j: int,
    weight: float,
) -> None:
    raw[i, j] += float(weight)
    raw[j, i] += float(weight)


def centrality_from_A(A: np.ndarray) -> np.ndarray:
    v = compute_eigenvector_centrality(
        compute_laplacian(np.asarray(A, dtype=float))
    )
    v = np.asarray(v, dtype=float).reshape(-1)
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)

    if v.sum() < 0:
        v = -v

    v = np.maximum(v, 0.0)
    if v.sum() <= 1e-12:
        v = np.abs(v)

    if v.sum() <= 1e-12:
        return np.full(N, 1.0 / N)

    return v / v.sum()

## Original unstructured environment generator

In [5]:
def build_unstructured_adjacency(topology_seed: int) -> np.ndarray:
    # Use the same graph-family settings as EnvironmentFactory's original
    # randomized environments, but extract A once and then reuse it for every
    # propagation law.
    x_dummy = np.linspace(0.0, 1.0, N)

    graph_env = NetworkGraph(
        num_agents=N,
        graph_model="barabasi_albert",
        ba_m=2,
        ba_prune_max_frac=0.5,
        ba_qsc_tol=1e-8,
        ba_max_tries=500,
        max_u=MAX_U,
        budget=1000.0,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        seed=int(topology_seed),
        terminate_when_converged=False,
        dynamics_model="laplacian",
        initial_opinions=x_dummy,
        control_resistance=np.zeros(N, dtype=float),
    )

    A = np.asarray(graph_env.connectivity_matrix, dtype=float).copy()
    assert A.shape == (N, N)
    return A


def build_unstructured_x0(opinion_seed: int) -> np.ndarray:
    base = np.linspace(0.0, 1.0, N)
    rng = np.random.default_rng(920_000 + int(opinion_seed))
    return rng.permutation(base).astype(float)


UNSTRUCTURED_A_BY_SEED = {
    int(seed): build_unstructured_adjacency(int(seed))
    for seed in UNSTRUCTURED_TOPOLOGY_SEEDS
}

print("Built unstructured graph seeds:", sorted(UNSTRUCTURED_A_BY_SEED))

Built unstructured graph seeds: [3, 4, 5, 6, 7]


## Structured mechanism generators

In [6]:
HK_LOW_CLUSTER = tuple(range(0, 7))
HK_HIGH_CLUSTER = tuple(range(7, 15))
HK_LOW_HUB = 0
HK_HIGH_HUB = 7

BROADCAST_HUBS = (0, 1)
BROADCAST_LEAVES = tuple(range(2, N))
BROADCAST_GROUP_A = tuple(range(2, 8))
BROADCAST_GROUP_B = tuple(range(8, N))


def build_hk_graph(
    family_name: str,
    environment_seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    params = HK_FAMILIES[family_name]
    raw = np.zeros((N, N), dtype=float)

    for node in HK_LOW_CLUSTER:
        if node != HK_LOW_HUB:
            _add_undirected(
                raw,
                HK_LOW_HUB,
                node,
                params["hub_leaf_weight"],
            )

    for node in HK_HIGH_CLUSTER:
        if node != HK_HIGH_HUB:
            _add_undirected(
                raw,
                HK_HIGH_HUB,
                node,
                params["hub_leaf_weight"],
            )

    for cluster, hub in [
        (HK_LOW_CLUSTER, HK_LOW_HUB),
        (HK_HIGH_CLUSTER, HK_HIGH_HUB),
    ]:
        leaves = [node for node in cluster if node != hub]
        pattern = params["peer_pattern"]
        w = float(params["peer_weight"])

        if pattern == "ring":
            for left, right in zip(
                leaves,
                leaves[1:] + leaves[:1],
            ):
                _add_undirected(raw, left, right, w)

        elif pattern == "paired":
            for index in range(0, len(leaves) - 1, 2):
                _add_undirected(
                    raw,
                    leaves[index],
                    leaves[index + 1],
                    w,
                )
            if len(leaves) % 2 == 1:
                _add_undirected(
                    raw,
                    leaves[-1],
                    leaves[0],
                    0.5 * w,
                )

        elif pattern == "dense":
            for pos, left in enumerate(leaves):
                for right in leaves[pos + 1:]:
                    _add_undirected(raw, left, right, w)

        else:
            raise ValueError(pattern)

    # The sole inter-community topological edge.
    _add_undirected(
        raw,
        HK_LOW_HUB,
        HK_HIGH_HUB,
        params["bridge_weight"],
    )

    rng = np.random.default_rng(
        10_000
        + 100 * list(HK_FAMILIES).index(family_name)
        + int(environment_seed)
    )
    raw = _jitter_positive_edges(
        raw,
        rng,
        EDGE_WEIGHT_JITTER_FRAC,
    )
    return raw, _normalize_rows(raw)


def build_hk_x0(environment_seed: int) -> np.ndarray:
    base = np.array(
        [
            0.35,
            0.27, 0.28, 0.29, 0.30, 0.31, 0.32,
            0.64,
            0.65, 0.66, 0.67, 0.68, 0.69, 0.70, 0.71,
        ],
        dtype=float,
    )
    rng = np.random.default_rng(20_000 + int(environment_seed))
    return np.clip(
        base
        + rng.uniform(
            -HK_OPINION_JITTER,
            HK_OPINION_JITTER,
            size=N,
        ),
        0.0,
        1.0,
    )


def build_broadcaster_graph(
    family_name: str,
    environment_seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    params = BROADCAST_FAMILIES[family_name]
    h = float(params["hub_influence_total"])
    kind = params["kind"]

    raw = np.zeros((N, N), dtype=float)
    leaves = list(BROADCAST_LEAVES)

    for idx, leaf in enumerate(leaves):
        prev_leaf = leaves[(idx - 1) % len(leaves)]
        next_leaf = leaves[(idx + 1) % len(leaves)]

        if kind == "dual":
            raw[leaf, 0] = 0.50 * h
            raw[leaf, 1] = 0.50 * h

        elif kind == "split":
            primary = 0 if leaf in BROADCAST_GROUP_A else 1
            secondary = 1 if primary == 0 else 0
            raw[leaf, primary] = 0.85 * h
            raw[leaf, secondary] = 0.15 * h

        elif kind == "asymmetric":
            raw[leaf, 0] = 0.75 * h
            raw[leaf, 1] = 0.25 * h

        else:
            raise ValueError(kind)

        remaining = 1.0 - h
        raw[leaf, prev_leaf] += 0.50 * remaining
        raw[leaf, next_leaf] += 0.50 * remaining

    raw[0, 1] = 0.20
    raw[1, 0] = 0.20

    for leaf in BROADCAST_GROUP_A:
        raw[0, leaf] += 0.80 / len(BROADCAST_GROUP_A)
    for leaf in BROADCAST_GROUP_B:
        raw[1, leaf] += 0.80 / len(BROADCAST_GROUP_B)

    rng = np.random.default_rng(
        30_000
        + 100 * list(BROADCAST_FAMILIES).index(family_name)
        + int(environment_seed)
    )
    raw = _jitter_positive_edges(
        raw,
        rng,
        EDGE_WEIGHT_JITTER_FRAC,
    )
    return raw, _normalize_rows(raw)


def build_broadcast_x0(
    *,
    hub_level: float,
    leaf_level: float,
    environment_seed: int,
) -> np.ndarray:
    x0 = np.empty(N, dtype=float)

    x0[0] = float(hub_level) - 0.01
    x0[1] = float(hub_level) + 0.01

    offsets = np.linspace(
        -0.035,
        0.035,
        len(BROADCAST_LEAVES),
    )
    x0[list(BROADCAST_LEAVES)] = float(leaf_level) + offsets

    rng = np.random.default_rng(40_000 + int(environment_seed))
    x0 += rng.uniform(
        -BROADCAST_OPINION_JITTER,
        BROADCAST_OPINION_JITTER,
        size=N,
    )

    return np.clip(x0, 0.0, 1.0)

## Build the complete fixed environment registry

In [7]:
ENVIRONMENT_SPECS: list[dict[str, Any]] = []

# ---------------------------------------------------------------------------
# Unstructured: all six dynamics, same graph/opinion seed grid.
# ---------------------------------------------------------------------------
for dynamics, dynamics_params in UNSTRUCTURED_DYNAMICS.items():
    for topology_seed in UNSTRUCTURED_TOPOLOGY_SEEDS:
        for opinion_seed in UNSTRUCTURED_OPINION_SEEDS:
            A = np.asarray(
                UNSTRUCTURED_A_BY_SEED[int(topology_seed)],
                dtype=float,
            ).copy()
            x0 = build_unstructured_x0(int(opinion_seed))

            ENVIRONMENT_SPECS.append(
                {
                    "environment_index": len(ENVIRONMENT_SPECS),
                    "environment_id": (
                        f"unstructured__{dynamics}"
                        f"__topo{int(topology_seed):02d}"
                        f"__x{int(opinion_seed):02d}"
                    ),
                    "scenario_class": "unstructured_random",
                    "family": "barabasi_albert_spread",
                    "dynamics": dynamics,
                    "topology_seed": int(topology_seed),
                    "opinion_seed": int(opinion_seed),
                    "environment_seed": None,
                    "dynamics_params": dict(dynamics_params),
                    "A": A,
                    "x0": x0,
                    "v_true": centrality_from_A(A),
                }
            )

# ---------------------------------------------------------------------------
# Structured HK.
# ---------------------------------------------------------------------------
for family in HK_FAMILIES:
    for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
        raw, A = build_hk_graph(family, environment_seed)
        x0 = build_hk_x0(environment_seed)

        ENVIRONMENT_SPECS.append(
            {
                "environment_index": len(ENVIRONMENT_SPECS),
                "environment_id": (
                    f"structured__hegselmannkrause"
                    f"__{family}__env{environment_seed:02d}"
                ),
                "scenario_class": "structured_mechanism",
                "family": family,
                "dynamics": "hegselmannkrause",
                "topology_seed": None,
                "opinion_seed": None,
                "environment_seed": int(environment_seed),
                "dynamics_params": {
                    "hk_epsilon": STRUCTURED_HK_EPSILON,
                    "hk_include_self": True,
                },
                "raw": raw,
                "A": A,
                "x0": x0,
                "v_true": centrality_from_A(A),
            }
        )

# ---------------------------------------------------------------------------
# Structured FJ / nonlinear influence.
# ---------------------------------------------------------------------------
for dynamics in ["friedkinjohnsen", "nonlinearinfluence"]:
    for family in BROADCAST_FAMILIES:
        for environment_seed in STRUCTURED_ENVIRONMENT_SEEDS:
            raw, A = build_broadcaster_graph(
                family,
                environment_seed,
            )

            if dynamics == "friedkinjohnsen":
                x0 = build_broadcast_x0(
                    hub_level=FJ_HUB_LEVEL,
                    leaf_level=FJ_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {
                    "fj_lambda": STRUCTURED_FJ_LAMBDA,
                }
            else:
                x0 = build_broadcast_x0(
                    hub_level=NLI_HUB_LEVEL,
                    leaf_level=NLI_LEAF_LEVEL,
                    environment_seed=environment_seed,
                )
                dynamics_params = {
                    "nonlinear_beta": STRUCTURED_NLI_BETA,
                }

            ENVIRONMENT_SPECS.append(
                {
                    "environment_index": len(ENVIRONMENT_SPECS),
                    "environment_id": (
                        f"structured__{dynamics}"
                        f"__{family}__env{environment_seed:02d}"
                    ),
                    "scenario_class": "structured_mechanism",
                    "family": family,
                    "dynamics": dynamics,
                    "topology_seed": None,
                    "opinion_seed": None,
                    "environment_seed": int(environment_seed),
                    "dynamics_params": dynamics_params,
                    "raw": raw,
                    "A": A,
                    "x0": x0,
                    "v_true": centrality_from_A(A),
                }
            )

assert len(ENVIRONMENT_SPECS) == 195
assert len({spec["environment_id"] for spec in ENVIRONMENT_SPECS}) == 195

registry_rows = [
    {
        key: spec.get(key)
        for key in [
            "environment_index",
            "environment_id",
            "scenario_class",
            "family",
            "dynamics",
            "topology_seed",
            "opinion_seed",
            "environment_seed",
        ]
    }
    for spec in ENVIRONMENT_SPECS
]

registry_df = pd.DataFrame(registry_rows)

display(
    registry_df.groupby(
        ["scenario_class", "dynamics"],
        as_index=False,
    ).size()
)

print("Distinct environments:", len(ENVIRONMENT_SPECS))
print("Learned runs:", len(ENVIRONMENT_SPECS) * len(LEARNING_SEEDS))

,scenario_class,dynamics,size
0,structured_mechanism,friedkinjohnsen,15
1,structured_mechanism,hegselmannkrause,15
2,structured_mechanism,nonlinearinfluence,15
3,unstructured_random,coca,25
4,unstructured_random,friedkinjohnsen,25
5,unstructured_random,hegselmannkrause,25
6,unstructured_random,laplacian,25
7,unstructured_random,nonlinearinfluence,25
8,unstructured_random,repulsion,25


Distinct environments: 195
Learned runs: 585


## Environment construction and policies

In [8]:
def make_env(
    spec: dict[str, Any],
    *,
    seed: int,
) -> NetworkGraph:
    params = dict(spec["dynamics_params"])

    kwargs = dict(
        connectivity_matrix=np.asarray(spec["A"], dtype=float).copy(),
        num_agents=N,
        max_u=MAX_U,
        desired_opinion=TARGET,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        initial_opinions=np.asarray(spec["x0"], dtype=float).copy(),
        control_resistance=np.zeros(N, dtype=float),
        dynamics_model=spec["dynamics"],
        budget=1000.0,
        max_steps=NUM_CAMPAIGNS + 20,
        opinion_end_tolerance=0.01,
        control_beta=0.4,
        normalize_reward=True,
        terminal_reward=0.0,
        terminate_when_converged=False,
        seed=int(seed),
    )

    if spec["dynamics"] == "hegselmannkrause":
        kwargs["hk_epsilon"] = float(params["hk_epsilon"])
        kwargs["hk_include_self"] = bool(params["hk_include_self"])

    elif spec["dynamics"] == "friedkinjohnsen":
        kwargs["fj_lambda"] = float(params["fj_lambda"])
        kwargs["fj_prejudice"] = np.asarray(
            spec["x0"],
            dtype=float,
        ).copy()

    elif spec["dynamics"] == "nonlinearinfluence":
        kwargs["nonlinear_beta"] = float(
            params["nonlinear_beta"]
        )

    elif spec["dynamics"] == "repulsion":
        kwargs["repulsion_epsilon"] = float(
            params["repulsion_epsilon"]
        )
        kwargs["repulsion_strength"] = float(
            params["repulsion_strength"]
        )

    return NetworkGraph(**kwargs)


def set_state(env: NetworkGraph, x0: np.ndarray) -> None:
    env.reset()
    env.opinions = np.asarray(x0, dtype=float).copy()

    if hasattr(env, "state"):
        try:
            env.state = np.asarray(x0, dtype=float).copy()
        except Exception:
            pass


def uniform_action(
    max_u: np.ndarray,
    budget: float,
) -> np.ndarray:
    return online_single_shot_module.uniform_budget_action(
        np.asarray(max_u, dtype=float).reshape(-1),
        float(budget),
    )


def true_graph_action(
    env: NetworkGraph,
    v_true: np.ndarray,
) -> np.ndarray:
    action, _ = centrality_based_continuous_control(
        env,
        float(B_CAMPAIGN),
        v=np.asarray(v_true, dtype=float),
    )
    return np.asarray(action, dtype=float)


def rollout_fixed_policy(
    spec: dict[str, Any],
    *,
    policy: str,
) -> dict[str, Any]:
    env = make_env(
        spec,
        seed=500_000 + int(spec["environment_index"]),
    )
    set_state(env, spec["x0"])

    states = [np.asarray(spec["x0"], dtype=float).copy()]
    actions = []
    rewards = []
    intermediate_states_list = []

    max_u_vec = np.asarray(env.max_u, dtype=float).reshape(-1)

    for campaign in range(NUM_CAMPAIGNS):
        if campaign == 0 or policy == "no_control":
            action = np.zeros(N, dtype=float)

        elif policy == "uniform":
            action = uniform_action(max_u_vec, B_CAMPAIGN)

        elif policy == "true_graph":
            action = true_graph_action(env, spec["v_true"])

        else:
            raise ValueError(policy)

        x_next, reward, done, trunc, info = env.step(action)

        states.append(np.asarray(x_next, dtype=float).copy())
        actions.append(np.asarray(action, dtype=float).copy())
        rewards.append(float(reward))
        intermediate_states_list.append(
            np.asarray(info["intermediate_states"], dtype=float).copy()
        )

        if done or trunc:
            break

    return {
        "policy": policy,
        "states": np.asarray(states, dtype=float),
        "actions": np.asarray(actions, dtype=float),
        "rewards": np.asarray(rewards, dtype=float),
        "intermediate_states_list": intermediate_states_list,
    }

## Structured-mechanism preflight

In [9]:
def active_component_labels(
    adjacency: np.ndarray,
    state: np.ndarray,
    epsilon: float,
) -> np.ndarray:
    state = np.asarray(state, dtype=float)

    undirected = (adjacency > 0) | (adjacency.T > 0)
    close = (
        np.abs(state[:, None] - state[None, :])
        <= float(epsilon)
    )
    active = undirected & close
    np.fill_diagonal(active, False)

    labels = np.full(N, -1, dtype=int)
    component = 0

    for start in range(N):
        if labels[start] >= 0:
            continue

        stack = [start]
        labels[start] = component

        while stack:
            node = stack.pop()
            for neighbor in np.flatnonzero(active[node]):
                if labels[neighbor] < 0:
                    labels[neighbor] = component
                    stack.append(int(neighbor))

        component += 1

    return labels


def number_active_components(
    adjacency: np.ndarray,
    state: np.ndarray,
    epsilon: float,
) -> int:
    labels = active_component_labels(
        adjacency,
        state,
        epsilon,
    )
    return int(labels.max() + 1)


def apply_impulse(
    state: np.ndarray,
    action: np.ndarray,
) -> np.ndarray:
    state = np.asarray(state, dtype=float)
    action = np.asarray(action, dtype=float)
    return action * TARGET + (1.0 - action) * state


def preflight_spec(spec: dict[str, Any]) -> dict[str, Any]:
    A = np.asarray(spec["A"], dtype=float)
    x0 = np.asarray(spec["x0"], dtype=float)
    v_true = np.asarray(spec["v_true"], dtype=float)

    assert A.shape == (N, N)
    assert np.allclose(A.sum(axis=1), 1.0)
    assert np.allclose(np.diag(A), 0.0)
    assert np.all(np.isfinite(x0))
    assert np.all((x0 >= 0.0) & (x0 <= 1.0))

    result = {
        "environment_id": spec["environment_id"],
        "scenario_class": spec["scenario_class"],
        "family": spec["family"],
        "dynamics": spec["dynamics"],
        "topology_seed": spec.get("topology_seed"),
        "opinion_seed": spec.get("opinion_seed"),
        "environment_seed": spec.get("environment_seed"),
        "x0_min": float(x0.min()),
        "x0_max": float(x0.max()),
        "x0_std": float(x0.std()),
    }

    # Generic environments are NOT filtered by oracle or learned performance.
    if spec["scenario_class"] == "unstructured_random":
        assert x0.max() - x0.min() >= 0.95
        return result

    # Structured environments retain the mechanism-only checks used in the
    # dedicated confirmatory study.
    uniform_rollout = rollout_fixed_policy(spec, policy="uniform")
    oracle_rollout = rollout_fixed_policy(spec, policy="true_graph")

    uniform_final = float(uniform_rollout["states"][-1].mean())
    oracle_final = float(oracle_rollout["states"][-1].mean())
    oracle_gap = oracle_final - uniform_final

    result.update(
        uniform_final_mean=uniform_final,
        oracle_final_mean=oracle_final,
        oracle_gap_final=oracle_gap,
    )

    if spec["dynamics"] == "hegselmannkrause":
        raw = np.asarray(spec["raw"], dtype=float)

        cross_edges = []
        for i in HK_LOW_CLUSTER:
            for j in HK_HIGH_CLUSTER:
                if raw[i, j] > 0 or raw[j, i] > 0:
                    cross_edges.append((int(i), int(j)))

        assert cross_edges == [(HK_LOW_HUB, HK_HIGH_HUB)]

        top_two = set(np.argsort(v_true)[-2:].tolist())
        assert top_two == {HK_LOW_HUB, HK_HIGH_HUB}

        eps = float(spec["dynamics_params"]["hk_epsilon"])
        assert number_active_components(A, x0, eps) == 2

        passive_rollout = rollout_fixed_policy(
            spec,
            policy="no_control",
        )
        x_passive = np.asarray(
            passive_rollout["states"][1],
            dtype=float,
        )
        assert number_active_components(A, x_passive, eps) == 2

        passive_gap = float(
            abs(
                x_passive[HK_LOW_HUB]
                - x_passive[HK_HIGH_HUB]
            )
        )
        assert passive_gap > eps

        u_uniform = uniform_action(
            np.full(N, MAX_U),
            B_CAMPAIGN,
        )
        x_uniform_impulse = apply_impulse(
            x_passive,
            u_uniform,
        )
        assert (
            abs(
                x_uniform_impulse[HK_LOW_HUB]
                - x_uniform_impulse[HK_HIGH_HUB]
            )
            > eps
        )

        env_oracle = make_env(
            spec,
            seed=610_000 + int(spec["environment_index"]),
        )
        set_state(env_oracle, x_passive)
        u_oracle = true_graph_action(
            env_oracle,
            v_true,
        )
        x_oracle_impulse = apply_impulse(
            x_passive,
            u_oracle,
        )

        assert (
            abs(
                x_oracle_impulse[HK_LOW_HUB]
                - x_oracle_impulse[HK_HIGH_HUB]
            )
            <= eps
        )

    else:
        top_two = set(np.argsort(v_true)[-2:].tolist())
        assert top_two == set(BROADCAST_HUBS)

        env_oracle = make_env(
            spec,
            seed=620_000 + int(spec["environment_index"]),
        )
        set_state(env_oracle, x0)
        first_oracle = true_graph_action(
            env_oracle,
            v_true,
        )

        hub_fraction = float(
            first_oracle[list(BROADCAST_HUBS)].sum()
            / max(first_oracle.sum(), 1e-12)
        )
        assert hub_fraction >= 0.50
        assert oracle_gap > 0.0

        result["oracle_first_action_hub_fraction"] = hub_fraction

    return result


preflight_df = pd.DataFrame(
    [preflight_spec(spec) for spec in ENVIRONMENT_SPECS]
)

print("All 195 environments passed validity/mechanism preflight.")
display(
    preflight_df.groupby(
        ["scenario_class", "dynamics"],
        as_index=False,
    ).size()
)

All 195 environments passed validity/mechanism preflight.


,scenario_class,dynamics,size
0,structured_mechanism,friedkinjohnsen,15
1,structured_mechanism,hegselmannkrause,15
2,structured_mechanism,nonlinearinfluence,15
3,unstructured_random,coca,25
4,unstructured_random,friedkinjohnsen,25
5,unstructured_random,hegselmannkrause,25
6,unstructured_random,laplacian,25
7,unstructured_random,nonlinearinfluence,25
8,unstructured_random,repulsion,25


## Cache identity and run manifest

In [10]:
def _json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            str(key): _json_safe(val)
            for key, val in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [_json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        x = float(value)
        return x if np.isfinite(x) else None
    if isinstance(value, np.bool_):
        return bool(value)
    if isinstance(value, float):
        return value if np.isfinite(value) else None
    return value


def sha256_json(payload: dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(
            _json_safe(payload),
            sort_keys=True,
            separators=(",", ":"),
        ).encode("utf-8")
    ).hexdigest()


def source_hash(obj: Any) -> str:
    path_string = inspect.getsourcefile(obj)
    if not path_string:
        return "unavailable"

    path = Path(path_string)
    if not path.exists():
        return "missing"

    return hashlib.sha256(path.read_bytes()).hexdigest()


SOURCE_HASHES = {
    "online_single_shot": source_hash(
        online_single_shot_module
    ),
    "NetworkGraph": source_hash(NetworkGraph),
}

SCIENTIFIC_CONFIG = {
    "pipeline_version": PIPELINE_VERSION,
    "N": N,
    "target": TARGET,
    "num_campaigns": NUM_CAMPAIGNS,
    "t_campaign": T_CAMPAIGN,
    "t_s": T_S,
    "max_u": MAX_U,
    "total_controlled_budget": TOTAL_CONTROLLED_BUDGET,
    "B_campaign": B_CAMPAIGN,
    "lambda_mix": LAMBDA_MIX,
    "exploration_campaigns": EXPLORATION_CAMPAIGNS,
    "epsilon_schedule": EPSILON_SCHEDULE,
    "fit_lr": FIT_LR,
    "fit_max_steps": FIT_MAX_STEPS,
    "fit_mae_stop": FIT_MAE_STOP,
    "fit_batch_size": FIT_BATCH_SIZE,
    "fit_check_every": FIT_CHECK_EVERY,
    "identifier_kwargs": IDENTIFIER_KWARGS,
    "learning_seeds": LEARNING_SEEDS,
    "unstructured_topology_seeds": UNSTRUCTURED_TOPOLOGY_SEEDS,
    "unstructured_opinion_seeds": UNSTRUCTURED_OPINION_SEEDS,
    "unstructured_dynamics": UNSTRUCTURED_DYNAMICS,
    "structured_environment_seeds": STRUCTURED_ENVIRONMENT_SEEDS,
    "hk_families": HK_FAMILIES,
    "broadcast_families": BROADCAST_FAMILIES,
    "structured_hk_epsilon": STRUCTURED_HK_EPSILON,
    "structured_fj_lambda": STRUCTURED_FJ_LAMBDA,
    "structured_nli_beta": STRUCTURED_NLI_BETA,
    "fj_hub_level": FJ_HUB_LEVEL,
    "fj_leaf_level": FJ_LEAF_LEVEL,
    "nli_hub_level": NLI_HUB_LEVEL,
    "nli_leaf_level": NLI_LEAF_LEVEL,
    "edge_weight_jitter_frac": EDGE_WEIGHT_JITTER_FRAC,
    "hk_opinion_jitter": HK_OPINION_JITTER,
    "broadcast_opinion_jitter": BROADCAST_OPINION_JITTER,
}

CONFIG_HASH = sha256_json(SCIENTIFIC_CONFIG)
IMPLEMENTATION_HASH = sha256_json(SOURCE_HASHES)

print("Config hash:", CONFIG_HASH[:16])
print("Implementation hash:", IMPLEMENTATION_HASH[:16])


def environment_cache_dir(spec: dict[str, Any]) -> Path:
    payload = {
        "environment_id": spec["environment_id"],
        "config_hash": CONFIG_HASH,
        "implementation_hash": IMPLEMENTATION_HASH,
    }
    key = sha256_json(payload)

    return (
        CACHE_ROOT
        / f"{spec['environment_id']}__{key[:16]}"
    )


def learned_cache_dir(
    spec: dict[str, Any],
    learning_seed: int,
) -> Path:
    payload = {
        "environment_id": spec["environment_id"],
        "learning_seed": int(learning_seed),
        "config_hash": CONFIG_HASH,
        "implementation_hash": IMPLEMENTATION_HASH,
    }
    key = sha256_json(payload)

    return (
        environment_cache_dir(spec)
        / "learned"
        / f"seed_{int(learning_seed):02d}_{key[:16]}"
    )


environment_manifest_rows = []

for spec in ENVIRONMENT_SPECS:
    environment_manifest_rows.append(
        {
            "environment_index": int(spec["environment_index"]),
            "environment_id": spec["environment_id"],
            "scenario_class": spec["scenario_class"],
            "family": spec["family"],
            "dynamics": spec["dynamics"],
            "topology_seed": spec.get("topology_seed"),
            "opinion_seed": spec.get("opinion_seed"),
            "environment_seed": spec.get("environment_seed"),
            "dynamics_params": spec["dynamics_params"],
            "cache_dir": str(environment_cache_dir(spec)),
        }
    )

RUN_MANIFEST = {
    "study_name": STUDY_NAME,
    "pipeline_version": PIPELINE_VERSION,
    "git_commit": GIT_COMMIT,
    "config_hash": CONFIG_HASH,
    "implementation_hash": IMPLEMENTATION_HASH,
    "source_hashes": SOURCE_HASHES,
    "scientific_config": SCIENTIFIC_CONFIG,
    "cache_root": str(CACHE_ROOT),
    "results_dir": str(RESULTS_DIR),
    "n_environments": len(ENVIRONMENT_SPECS),
    "n_learning_seeds": len(LEARNING_SEEDS),
    "n_learned_runs_expected": (
        len(ENVIRONMENT_SPECS) * len(LEARNING_SEEDS)
    ),
    "learning_seeds": LEARNING_SEEDS,
    "environments": environment_manifest_rows,
}

(
    RESULTS_DIR / "RUN_MANIFEST.json"
).write_text(
    json.dumps(
        _json_safe(RUN_MANIFEST),
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

preflight_df.to_csv(
    RESULTS_DIR / "environment_preflight.csv",
    index=False,
)

print("Run manifest:", RESULTS_DIR / "RUN_MANIFEST.json")

Config hash: 110bb71b67ca12e2
Implementation hash: e37ff12c7c61152c
Run manifest: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\experiment_2026_08_17_broad_all_dynamics_multitopology_multiseed\RUN_MANIFEST.json


## Rollout metrics and cache I/O

In [11]:
def fine_trajectory(
    rollout: dict[str, Any],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    times = [0.0]
    states = [
        np.asarray(rollout["states"][0], dtype=float)
    ]
    campaigns = [-1]

    for campaign, inter in enumerate(
        rollout["intermediate_states_list"]
    ):
        inter = np.asarray(inter, dtype=float)
        t0 = campaign * T_CAMPAIGN

        times.append(t0)
        states.append(inter[0])
        campaigns.append(campaign)

        for step in range(1, len(inter)):
            times.append(t0 + step * T_S)
            states.append(inter[step])
            campaigns.append(campaign)

    return (
        np.asarray(times, dtype=float),
        np.asarray(states, dtype=float),
        np.asarray(campaigns, dtype=int),
    )


def base_metrics(
    rollout: dict[str, Any],
) -> dict[str, float]:
    states = np.asarray(rollout["states"], dtype=float)
    final = states[-1]
    mean_path = states.mean(axis=1)

    return {
        "final_mean": float(final.mean()),
        "mean_over_campaigns": float(mean_path.mean()),
        "final_min": float(final.min()),
        "final_max": float(final.max()),
        "final_mean_distance_to_target": float(
            np.mean(np.abs(TARGET - final))
        ),
    }


def hk_bridge_metrics(
    spec: dict[str, Any],
    rollout: dict[str, Any],
) -> dict[str, Any]:
    if (
        spec["scenario_class"] != "structured_mechanism"
        or spec["dynamics"] != "hegselmannkrause"
    ):
        return {}

    eps = float(spec["dynamics_params"]["hk_epsilon"])
    times, states, campaigns = fine_trajectory(rollout)

    gaps = np.abs(
        states[:, HK_LOW_HUB]
        - states[:, HK_HIGH_HUB]
    )
    active = gaps <= eps

    first_index = (
        int(np.flatnonzero(active)[0])
        if active.any()
        else None
    )

    components = np.array(
        [
            number_active_components(
                np.asarray(spec["A"], dtype=float),
                state,
                eps,
            )
            for state in states
        ],
        dtype=int,
    )
    connected = components == 1

    first_connected_index = (
        int(np.flatnonzero(connected)[0])
        if connected.any()
        else None
    )

    return {
        "first_bridge_time": (
            float(times[first_index])
            if first_index is not None
            else np.nan
        ),
        "first_bridge_campaign": (
            int(campaigns[first_index])
            if first_index is not None
            else np.nan
        ),
        "first_connected_time": (
            float(times[first_connected_index])
            if first_connected_index is not None
            else np.nan
        ),
        "first_connected_campaign": (
            int(campaigns[first_connected_index])
            if first_connected_index is not None
            else np.nan
        ),
        "final_hub_gap": float(gaps[-1]),
        "final_components": int(components[-1]),
    }


def mechanism_metrics(
    spec: dict[str, Any],
    rollout: dict[str, Any],
) -> dict[str, Any]:
    result = {}
    result.update(hk_bridge_metrics(spec, rollout))

    actions = np.asarray(rollout["actions"], dtype=float)

    if spec["scenario_class"] == "structured_mechanism":
        if spec["dynamics"] == "hegselmannkrause":
            hubs = [HK_LOW_HUB, HK_HIGH_HUB]
        else:
            hubs = list(BROADCAST_HUBS)

        if len(actions) > 1:
            controlled = actions[1:]
            total = controlled.sum(axis=1)
            hub_total = controlled[:, hubs].sum(axis=1)

            with np.errstate(divide="ignore", invalid="ignore"):
                fractions = np.where(
                    total > 1e-12,
                    hub_total / total,
                    np.nan,
                )

            result[
                "mean_controlled_hub_budget_fraction"
            ] = float(np.nanmean(fractions))

    return result


def summarize_rollout(
    spec: dict[str, Any],
    rollout: dict[str, Any],
) -> dict[str, Any]:
    return {
        **base_metrics(rollout),
        **mechanism_metrics(spec, rollout),
    }


def _save_rollouts_npz(
    path: Path,
    rollouts: dict[str, dict[str, Any]],
) -> None:
    arrays: dict[str, np.ndarray] = {}

    for policy, rollout in rollouts.items():
        arrays[f"{policy}__states"] = np.asarray(
            rollout["states"],
            dtype=float,
        )
        arrays[f"{policy}__actions"] = np.asarray(
            rollout["actions"],
            dtype=float,
        )

        for campaign, inter in enumerate(
            rollout["intermediate_states_list"]
        ):
            arrays[
                f"{policy}__intermediate__{campaign:02d}"
            ] = np.asarray(inter, dtype=float)

    temporary = path.with_suffix(".tmp")
    with temporary.open("wb") as handle:
        np.savez_compressed(handle, **arrays)

    os.replace(temporary, path)


def _load_policy_from_npz(
    arrays: Any,
    policy: str,
) -> dict[str, Any]:
    inter_keys = sorted(
        key
        for key in arrays.files
        if key.startswith(
            f"{policy}__intermediate__"
        )
    )

    return {
        "policy": policy,
        "states": arrays[f"{policy}__states"],
        "actions": arrays[f"{policy}__actions"],
        "intermediate_states_list": [
            arrays[key]
            for key in inter_keys
        ],
    }

In [12]:
def baseline_cache_complete(
    spec: dict[str, Any],
) -> bool:
    directory = environment_cache_dir(spec)

    required = [
        directory / "baseline_manifest.json",
        directory / "baselines.npz",
        directory / "baseline_summary.json",
    ]

    if not all(path.exists() for path in required):
        return False

    try:
        manifest = json.loads(
            (directory / "baseline_manifest.json").read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return False

    return (
        manifest.get("status") == "success"
        and manifest.get("config_hash") == CONFIG_HASH
        and manifest.get("implementation_hash") == IMPLEMENTATION_HASH
    )


def learned_cache_complete(
    spec: dict[str, Any],
    learning_seed: int,
) -> bool:
    directory = learned_cache_dir(
        spec,
        learning_seed,
    )

    required = [
        directory / "manifest.json",
        directory / "rollout.npz",
        directory / "summary.json",
    ]

    if not all(path.exists() for path in required):
        return False

    try:
        manifest = json.loads(
            (directory / "manifest.json").read_text(
                encoding="utf-8"
            )
        )
    except Exception:
        return False

    return (
        manifest.get("status") == "success"
        and manifest.get("config_hash") == CONFIG_HASH
        and manifest.get("implementation_hash") == IMPLEMENTATION_HASH
    )


def ensure_baseline_cache(
    spec: dict[str, Any],
) -> list[dict[str, Any]]:
    if baseline_cache_complete(spec):
        return json.loads(
            (
                environment_cache_dir(spec)
                / "baseline_summary.json"
            ).read_text(encoding="utf-8")
        )

    directory = environment_cache_dir(spec)
    directory.mkdir(parents=True, exist_ok=True)

    rollouts = {
        policy: rollout_fixed_policy(
            spec,
            policy=policy,
        )
        for policy in [
            "no_control",
            "uniform",
            "true_graph",
        ]
    }

    rows = []

    for policy, rollout in rollouts.items():
        rows.append(
            {
                "environment_id": spec["environment_id"],
                "scenario_class": spec["scenario_class"],
                "family": spec["family"],
                "dynamics": spec["dynamics"],
                "topology_seed": spec.get("topology_seed"),
                "opinion_seed": spec.get("opinion_seed"),
                "environment_seed": spec.get("environment_seed"),
                "policy": policy,
                **summarize_rollout(spec, rollout),
            }
        )

    _save_rollouts_npz(
        directory / "baselines.npz",
        rollouts,
    )

    (
        directory / "baseline_summary.json"
    ).write_text(
        json.dumps(_json_safe(rows), indent=2),
        encoding="utf-8",
    )

    temporary = directory / "baseline_manifest.json.tmp"
    final = directory / "baseline_manifest.json"

    temporary.write_text(
        json.dumps(
            {
                "status": "success",
                "environment_id": spec["environment_id"],
                "config_hash": CONFIG_HASH,
                "implementation_hash": IMPLEMENTATION_HASH,
                "git_commit": GIT_COMMIT,
                "completed_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            },
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )
    os.replace(temporary, final)

    return rows


def set_global_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def run_learned(
    spec: dict[str, Any],
    *,
    learning_seed: int,
) -> dict[str, Any]:
    environment_index = int(spec["environment_index"])

    train_seed = (
        700_000
        + 100 * environment_index
        + int(learning_seed)
    )
    rng_seed = (
        800_000
        + 100 * environment_index
        + int(learning_seed)
    )

    set_global_seed(train_seed)

    env_template = make_env(
        spec,
        seed=900_000 + environment_index,
    )

    result = run_single_shot_online_identification(
        env_template,
        x0=np.asarray(spec["x0"], dtype=float),
        random_initial_opinions=False,
        num_campaigns_total=NUM_CAMPAIGNS,
        t_campaign=T_CAMPAIGN,
        t_s=T_S,
        B_campaign=B_CAMPAIGN,
        lambda_mix=LAMBDA_MIX,
        exploration_campaigns=EXPLORATION_CAMPAIGNS,
        epsilon_schedule=EPSILON_SCHEDULE,
        lr=FIT_LR,
        l2_lambda=0.0,
        fit_max_steps=FIT_MAX_STEPS,
        fit_mae_stop=FIT_MAE_STOP,
        fit_batch_size=FIT_BATCH_SIZE,
        fit_check_every=FIT_CHECK_EVERY,
        identifier_kwargs=IDENTIFIER_KWARGS,
        device=DEVICE,
        rng_seed=rng_seed,
        suppress_fit_logs=True,
    )

    result = dict(result)
    result["policy"] = "learned_nonlinear"
    result["train_seed"] = train_seed
    result["rng_seed"] = rng_seed
    return result


def ensure_learned_cache(
    spec: dict[str, Any],
    *,
    learning_seed: int,
) -> dict[str, Any]:
    directory = learned_cache_dir(
        spec,
        learning_seed,
    )

    if learned_cache_complete(spec, learning_seed):
        return json.loads(
            (directory / "summary.json").read_text(
                encoding="utf-8"
            )
        )

    directory.mkdir(parents=True, exist_ok=True)

    rollout = run_learned(
        spec,
        learning_seed=learning_seed,
    )

    summary = {
        "environment_id": spec["environment_id"],
        "scenario_class": spec["scenario_class"],
        "family": spec["family"],
        "dynamics": spec["dynamics"],
        "topology_seed": spec.get("topology_seed"),
        "opinion_seed": spec.get("opinion_seed"),
        "environment_seed": spec.get("environment_seed"),
        "learning_seed": int(learning_seed),
        "train_seed": int(rollout["train_seed"]),
        "rng_seed": int(rollout["rng_seed"]),
        "policy": "learned_nonlinear",
        **summarize_rollout(spec, rollout),
    }

    fit_infos = rollout.get("fit_infos", [])
    if fit_infos:
        last_fit = fit_infos[-1]
        summary.update(
            final_fit_train_mae=last_fit.get(
                "train_mae",
                np.nan,
            ),
            final_fit_identity_mae=last_fit.get(
                "identity_mae",
                np.nan,
            ),
            final_fit_model_over_identity=last_fit.get(
                "model_over_identity",
                np.nan,
            ),
            final_fit_n_pairs=last_fit.get(
                "n_pairs",
                np.nan,
            ),
        )

    _save_rollouts_npz(
        directory / "rollout.npz",
        {"learned_nonlinear": rollout},
    )

    (
        directory / "summary.json"
    ).write_text(
        json.dumps(_json_safe(summary), indent=2),
        encoding="utf-8",
    )

    temporary = directory / "manifest.json.tmp"
    final = directory / "manifest.json"

    temporary.write_text(
        json.dumps(
            {
                "status": "success",
                "environment_id": spec["environment_id"],
                "learning_seed": int(learning_seed),
                "config_hash": CONFIG_HASH,
                "implementation_hash": IMPLEMENTATION_HASH,
                "git_commit": GIT_COMMIT,
                "completed_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            },
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )
    os.replace(temporary, final)

    return summary

## Parallel worker

In [13]:
if WORKER_MODE:
    assigned_specs = [
        spec
        for spec in ENVIRONMENT_SPECS
        if int(spec["environment_index"]) % NUM_SHARDS == SHARD_ID
    ]

    print("Assigned environments:", len(assigned_specs))

    shard_rows = []
    t0 = time.perf_counter()

    for position, spec in enumerate(
        assigned_specs,
        start=1,
    ):
        print(
            f"\n[{position}/{len(assigned_specs)}] "
            f"{spec['environment_id']}",
            flush=True,
        )

        ensure_baseline_cache(spec)

        for learning_seed in LEARNING_SEEDS:
            print(
                f"  learned seed {learning_seed}",
                flush=True,
            )

            summary = ensure_learned_cache(
                spec,
                learning_seed=learning_seed,
            )
            shard_rows.append(summary)

    elapsed = time.perf_counter() - t0

    pd.DataFrame(shard_rows).to_csv(
        RESULTS_DIR / f"worker_shard_{SHARD_ID}_summary.csv",
        index=False,
    )

    (
        RESULTS_DIR / f"SHARD_{SHARD_ID}_COMPLETE.json"
    ).write_text(
        json.dumps(
            {
                "status": "success",
                "shard_id": SHARD_ID,
                "n_environments": len(assigned_specs),
                "n_learned_rows": len(shard_rows),
                "elapsed_s": float(elapsed),
                "config_hash": CONFIG_HASH,
                "implementation_hash": IMPLEMENTATION_HASH,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        f"\nShard {SHARD_ID} complete in "
        f"{elapsed / 60:.1f} minutes."
    )

else:
    print(
        "Check mode: this notebook will not launch expensive training "
        "without BROAD_FULL_SHARD_ID."
    )

Check mode: this notebook will not launch expensive training without BROAD_FULL_SHARD_ID.


## Cache completeness check

In [14]:
missing_baselines = [
    spec["environment_id"]
    for spec in ENVIRONMENT_SPECS
    if not baseline_cache_complete(spec)
]

missing_learned = [
    (spec["environment_id"], learning_seed)
    for spec in ENVIRONMENT_SPECS
    for learning_seed in LEARNING_SEEDS
    if not learned_cache_complete(
        spec,
        learning_seed,
    )
]

print("Expected environments:", len(ENVIRONMENT_SPECS))
print(
    "Expected learned runs:",
    len(ENVIRONMENT_SPECS) * len(LEARNING_SEEDS),
)
print("Missing baseline environments:", len(missing_baselines))
print("Missing learned runs:", len(missing_learned))

if not missing_baselines and not missing_learned:
    print("Complete broad-experiment cache validated.")
    (
        RESULTS_DIR / "COMPUTATION_COMPLETE.json"
    ).write_text(
        json.dumps(
            {
                "status": "success",
                "study_name": STUDY_NAME,
                "git_commit": GIT_COMMIT,
                "config_hash": CONFIG_HASH,
                "implementation_hash": IMPLEMENTATION_HASH,
                "n_environments": len(ENVIRONMENT_SPECS),
                "n_learned_runs": (
                    len(ENVIRONMENT_SPECS)
                    * len(LEARNING_SEEDS)
                ),
                "completed_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),
            },
            indent=2,
        ),
        encoding="utf-8",
    )
else:
    print("First missing baselines:", missing_baselines[:5])
    print("First missing learned:", missing_learned[:5])
    if not WORKER_MODE:
        print(
            "Run/re-run the parallel launcher. Completed entries "
            "will be reused from the persistent cache."
        )

Expected environments: 195
Expected learned runs: 585
Missing baseline environments: 0
Missing learned runs: 0
Complete broad-experiment cache validated.
